# Sentence Similarity

> Everything to know about comparing two texts: the bi-encoder / cross-encoder split and what it costs, why Spearman is the STS metric, why a cosine of 0.8 means nothing on its own, and runnable code that scores four systems on STS-B and then uses them to deduplicate a corpus.

- skip_showdoc: true
- skip_exec: true

## 1. What is Sentence Similarity?

Sentence similarity scores how alike two texts are in meaning. `"A man is playing a guitar"` and `"A person plays an instrument"` are similar; `"A man is playing a guitar"` and `"A man is cleaning a guitar"` share more words and are less similar. That gap - overlap versus meaning - is the entire task.

**Two architectures, and choosing between them is the decision that matters:**

| | **Bi-encoder** | **Cross-encoder** |
|---|---|---|
| How | Encode each text separately, compare vectors | Encode the **pair** together, output one score |
| Cost for n texts, all pairs | n encodings + n^2 cheap dot products | **n^2 forward passes** |
| Precomputable | Yes - index once, compare forever | No - the score exists only for that pair |
| Accuracy | Good | Consistently better, typically several points of correlation |
| Use for | Search, clustering, dedup at scale | Reranking a shortlist, high-stakes pairwise decisions |

The reason both exist: a bi-encoder never lets the two texts see each other, so it must compress each into a vector that is useful against *any* future partner. A cross-encoder reads them jointly with full attention across both, which is strictly more informative and strictly more expensive. The standard production shape is bi-encoder to shortlist, cross-encoder to decide - see `11_Text_Ranking`.

**Symmetric vs asymmetric**, the other distinction people get wrong:

- **Symmetric**: both sides are the same kind of text ("are these two questions duplicates?"). STS, dedup, clustering, paraphrase mining. This notebook.
- **Asymmetric**: a short query against a long passage ("which document answers this?"). Retrieval. `07_Feature_Extraction` and `11_Text_Ranking`.

Models are tuned for one or the other, and many expect a prefix that tells them which mode they are in. Using a retrieval-tuned model for symmetric comparison without its prefix quietly loses accuracy.

---

## 2. Real-World Use Cases

| Use case | Domain | Consumes / produces | Dominant constraint |
|---|---|---|---|
| Duplicate detection | Forums, issue trackers, CRM | New item -> existing near-duplicates | Threshold calibration; a wrong merge is worse than a miss |
| Semantic caching for LLMs | Any LLM product | Incoming prompt -> cached answer if close enough | Threshold; a false hit serves the wrong answer |
| Support ticket clustering | Customer service | Tickets -> themes | Unknown cluster count; human-readable labels |
| Plagiarism and content reuse | Education, publishing | Document -> matching sources | Paraphrase robustness; evidence for a human reviewer |
| Recommendation ("more like this") | Media, commerce | Item -> similar items | Cold start; diversity vs similarity |
| Data deduplication for training | ML teams | Corpus -> deduplicated corpus | Scale (billions of pairs); MinHash first, embeddings second |
| Test-suite deduplication | QA, evaluation | Test cases -> redundant ones | Precision; removing a unique test is silent damage |
| Entity and record matching | Data engineering | Two records -> same entity? | Structured fields matter as much as text |

What the leaderboard number hides:

- **A cosine score is not a probability and is not portable.** Model A's 0.85 and model B's 0.85 are unrelated, and neither is "85% similar". Every deployment needs its own threshold, chosen on its own labelled pairs.
- **Embedding spaces are anisotropic.** Random sentence pairs from most models cluster around a high baseline cosine (often 0.3-0.6, sometimes higher), so "0.5 means half similar" is wrong in a specific and misleading way. Look at the *distribution* before picking a cut - section 10 plots it.
- **Negation and small edits are a known blind spot.** "The deploy succeeded" and "the deploy did not succeed" usually score very high. If your task hinges on that distinction, a bi-encoder alone will not do it.
- **All-pairs is quadratic.** A million documents is 500 billion pairs. Real dedup uses blocking (MinHash/LSH, or an ANN index) to generate candidates and reserves the expensive scorer for those.

---

## 3. How Modern Sentence Similarity Works

1. **Lexical overlap (pre-2013).** Jaccard, TF-IDF cosine, edit distance. Fast, exact, and blind to paraphrase - still the right tool for near-exact duplicates, and what MinHash/LSH scales to billions.
2. **Averaged word vectors (2013-2018).** Mean of word2vec/GloVe vectors. Captures topical similarity, ignores word order and negation.
3. **BERT, used wrongly (2018-2019).** Mean-pooling raw BERT for similarity performs *worse than averaged GloVe* on STS. BERT was never trained to make similar sentences have nearby vectors, and this surprised a lot of people.
4. **Sentence-BERT (2019).** Fine-tune BERT with a siamese network on NLI pairs: entailed pairs pulled together, contradictions pushed apart. This made bi-encoder similarity work, and reduced a 65-hour all-pairs cross-encoder job over 10k sentences to about 5 seconds of vector comparison.
5. **Contrastive self-supervision (2021).** SimCSE: encode the same sentence twice with different dropout masks and treat the pair as positives. Almost free positives, and it also fixed much of the anisotropy that made raw BERT vectors cluster.
6. **Scale and instructions (2022-2024).** E5, GTE, BGE: hundreds of millions of weakly-supervised pairs plus labelled fine-tuning, with prefixes that switch a single model between symmetric and asymmetric behaviour.
7. **LLM-based embedders and Matryoshka (2023-2026).** Decoder models converted into embedders (Qwen3-Embedding, gte-Qwen, E5-Mistral) lead MTEB; Matryoshka training makes the leading dimensions independently usable. See `07_Feature_Extraction` for the storage arithmetic.
8. **Cross-encoders never went away.** They remain the accuracy ceiling for pairwise scoring, and the 2024-2026 development is LLM-based rerankers that read both texts and emit a relevance judgement - the same idea with a much larger reader.

**Where it stands (mid-2026).** Bi-encoders are the default for anything at scale and a 100-150M contrastive encoder is enough for most symmetric work. Cross-encoders are for the shortlist. The genuinely hard part of a deployment is not the model - it is choosing the threshold and handling the negation and numeric edge cases.

---

## 4. Evaluation Metrics

**Spearman rank correlation** is the STS standard. It compares the *ranking* of predicted similarities against the ranking of human ratings, so it does not care that one model's scores live in [0.3, 0.95] and another's in [-0.1, 0.9]. That scale-invariance is exactly right for a metric over uncalibrated similarity scores, and it is why Spearman rather than Pearson is reported.

$$\rho = \text{Pearson}\big(\text{rank}(x), \text{rank}(y)\big)$$

**Pearson** is also reported sometimes; it assumes a *linear* relationship between the score and human judgement, which is a stronger assumption than any embedding model earns.

**For a deployed system, correlation is the wrong metric.** What you ship is a decision - duplicate or not - so measure **precision/recall/F1 at the threshold you will actually use**, and choose that threshold from the cost of each error. A model with slightly lower Spearman but a cleaner separation at your operating point is the better model.

**Pitfalls:**

- **STS-B is short, clean, image-caption-derived English.** It correlates with general quality and predicts nothing about your ticket titles or legal clauses.
- **Human ratings disagree** by roughly 0.1-0.2 correlation between annotators, which caps what any model can score.
- **Never compare raw score distributions across models.** Compare rankings (Spearman) or per-model calibrated thresholds.

The cell below implements Spearman, Pearson, and a threshold sweep that reports F1 at each cut - the pair of instruments this notebook uses throughout.

---

In [ ]:
def rankdata(values):
    "Ranks with ties averaged - the standard definition Spearman needs."
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        mean_rank = (i + j) / 2 + 1
        for k in range(i, j + 1):
            ranks[order[k]] = mean_rank
        i = j + 1
    return ranks


def pearson(x, y):
    "Pearson product-moment correlation."
    n = len(x)
    mx, my = sum(x) / n, sum(y) / n
    cov = sum((a - mx) * (b - my) for a, b in zip(x, y))
    vx = sum((a - mx) ** 2 for a in x) ** 0.5
    vy = sum((b - my) ** 2 for b in y) ** 0.5
    return cov / (vx * vy) if vx and vy else 0.0


def spearman(x, y):
    "Rank correlation - scale-invariant, which is what an uncalibrated similarity needs."
    return pearson(rankdata(list(x)), rankdata(list(y)))


def threshold_sweep(scores, labels, steps=41):
    "Precision/recall/F1 at every candidate cut - how you actually pick an operating point."
    lo, hi = min(scores), max(scores)
    rows = []
    for i in range(steps):
        thr = lo + (hi - lo) * i / (steps - 1)
        tp = sum(s >= thr and l for s, l in zip(scores, labels))
        fp = sum(s >= thr and not l for s, l in zip(scores, labels))
        fn = sum(s < thr and l for s, l in zip(scores, labels))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        rows.append({"threshold": thr, "precision": prec, "recall": rec, "f1": f1})
    return rows


# Sanity check: a monotone but non-linear relation. Spearman sees it perfectly, Pearson does not.
human = [1, 2, 3, 4, 5, 6, 7, 8]
model = [v ** 3 for v in human]
print(f"perfectly monotone, non-linear:  spearman {spearman(human, model):.3f}  "
      f"pearson {pearson(human, model):.3f}")

noisy = [1, 3, 2, 4, 6, 5, 8, 7]
print(f"same ranking with adjacent swaps: spearman {spearman(human, noisy):.3f}  "
      f"pearson {pearson(human, noisy):.3f}")

labels = [1, 1, 1, 0, 0, 0]
scores = [0.9, 0.8, 0.55, 0.5, 0.3, 0.2]
best = max(threshold_sweep(scores, labels), key=lambda r: r["f1"])
print(f"\nbest F1 {best['f1']:.2f} at threshold {best['threshold']:.2f} "
      f"(P {best['precision']:.2f} R {best['recall']:.2f})")

## 5. Datasets

| Dataset | Contents | Size | Scope | License | Typical use |
|---|---|---|---|---|---|
| [STS Benchmark](https://huggingface.co/datasets/sentence-transformers/stsb) | Sentence pairs, human similarity 0-5 | 5.7k / 1.5k / 1.4k | en | mixed | The reference symmetric benchmark; used below |
| [STS12-16 / SICK-R](https://huggingface.co/datasets/mteb/sickr-sts) | Earlier SemEval similarity sets | 1-10k each | en | mixed | The MTEB STS suite |
| [Quora Duplicate Questions](https://huggingface.co/datasets/sentence-transformers/quora-duplicates) | Question pairs, duplicate or not | 400k | en | non-commercial | Binary duplicate detection, threshold tuning |
| [PAWS](https://huggingface.co/datasets/google-research-datasets/paws) | High word-overlap pairs, adversarial | 108k | en (+ PAWS-X) | free for research | **Word overlap != meaning**; where naive models break |
| [MRPC](https://huggingface.co/datasets/nyu-mll/glue) | News sentence pairs, paraphrase or not | 5.8k | en | mixed | Classic paraphrase classification |
| [SNLI / MNLI](https://huggingface.co/datasets/nyu-mll/glue) | Premise/hypothesis with entailment labels | 1M | en | mixed | Sentence-BERT's training signal |
| [AllNLI triplets](https://huggingface.co/datasets/sentence-transformers/all-nli) | (anchor, positive, negative) | 940k | en | mixed | Contrastive fine-tuning |
| [STS17 / STS22](https://huggingface.co/datasets/mteb/sts22-crosslingual-sts) | Cross-lingual similarity | 250-3k per pair | 10+ langs | mixed | Multilingual and cross-lingual STS |

This notebook evaluates on **STS-B test** (1,379 pairs, scores already normalised to 0-1 in this copy). Two caveats to hold: the sentences are short and caption-like, and the benchmark is largely saturated, so differences of one or two correlation points are not meaningful. **PAWS** is the more revealing dataset if you want to see a model fail - it is built from pairs with near-identical wording and opposite meaning.

---

## 6. The Model Landscape (mid-2026)

The reference ranking is the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) STS tab, with the caveats from `07_Feature_Extraction` section 2.

| Model | Params | Dims | Type | License | Best for |
|---|---|---|---|---|---|
| [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | 22M | 384 | bi-encoder | Apache 2.0 | the CPU default for symmetric similarity; used below |
| [all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | 110M | 768 | bi-encoder | Apache 2.0 | the classic strong symmetric model |
| [bge-base-en-v1.5](https://huggingface.co/BAAI/bge-base-en-v1.5) | 109M | 768 | bi-encoder | MIT | retrieval-first, strong at STS too; used below |
| [gte-modernbert-base](https://huggingface.co/Alibaba-NLP/gte-modernbert-base) | 149M | 768 | bi-encoder | Apache 2.0 | long inputs (8192), fast |
| [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | 596M | 1024 | bi-encoder | Apache 2.0 | top-tier quality; used below |
| [stsb-roberta-base (cross)](https://huggingface.co/cross-encoder/stsb-roberta-base) | 125M | - | **cross-encoder** | Apache 2.0 | the accuracy ceiling for pairs; used below |
| [stsb-TinyBERT-L4 (cross)](https://huggingface.co/cross-encoder/stsb-TinyBERT-L4) | 14M | - | cross-encoder | Apache 2.0 | cheap cross-encoding |
| [paraphrase-multilingual-mpnet](https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2) | 278M | 768 | bi-encoder | Apache 2.0 | 50+ languages, symmetric |
| [multilingual-e5-large](https://huggingface.co/intfloat/multilingual-e5-large) | 560M | 1024 | bi-encoder | MIT | 100 languages |

**How to choose.** Comparing many texts to many texts: a bi-encoder, always. Deciding a handful of high-stakes pairs, or reranking a shortlist: a cross-encoder. Multilingual, including cross-lingual matching: a `paraphrase-multilingual-*` or `multilingual-e5` model - these place translations of the same sentence close together, which monolingual models do not.

---

## 7. Setup

Package roles:

- `transformers` (>=5.13) + `torch` - three bi-encoders and one cross-encoder
- `accelerate` - device placement
- `datasets` - STS-B test
- `pandas` + `pyecharts` - benchmark tables and charts

**No `sentence-transformers`.** These are plain `transformers` encoders; the pooling is done explicitly (section 8) because it is the thing worth understanding. In production the wrapper is a reasonable choice - it is applying the same code.

**How a cross-encoder is loaded.** It is an `AutoModelForSequenceClassification` with **one** output logit, fed `tokenizer(text_a, text_b)` so both texts share a sequence with a separator between them. The single logit is the similarity score. That is the whole architecture: no vectors, no index, and no way to precompute anything.

---

In [ ]:
# Everything runs through Hugging Face transformers - no sentence-transformers wrapper.
# %pip install -q torch transformers accelerate datasets pandas pyecharts

In [ ]:
import ctypes
import ctypes.util
import gc
import time
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device, "| dtype:", dtype)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:22s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "Collect garbage and hand freed VRAM back to the CUDA allocator.\n\n    Call right after `del`-ing a model you are done with: `del model; free_memory()`.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    # glibc keeps freed CPU allocations in its arenas instead of returning them to the
    # OS, so RSS compounds across sections. malloc_trim(0) hands the arenas back.
    try:
        ctypes.CDLL(ctypes.util.find_library("c") or "libc.so.6").malloc_trim(0)
    except Exception:
        pass


# All downloads go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)
HF_CACHE = str(DATA_DIR / "hf_cache")

In [ ]:
from datasets import load_dataset

stsb = load_dataset("sentence-transformers/stsb", split="test", cache_dir=HF_CACHE)

N = 600  # pairs to score (1379 in the full test split)
pairs = stsb.select(range(min(N, len(stsb))))
sent_a = [r["sentence1"] for r in pairs]
sent_b = [r["sentence2"] for r in pairs]
human = [float(r["score"]) for r in pairs]          # already normalised to 0-1 in this copy

# A binary view of the same data, for threshold and F1 work.
DUP_THRESHOLD = 0.8                                  # >=4/5 on the original scale
is_duplicate = [s >= DUP_THRESHOLD for s in human]

print(stsb)
print(f"\n{len(sent_a)} pairs, {sum(is_duplicate)} labelled near-duplicate at human >= {DUP_THRESHOLD}")
for a, b, s in list(zip(sent_a, sent_b, human))[:4]:
    print(f"\n  {s:.2f}  {a}\n        {b}")

## 8. The bi-encoder path

Encode each sentence once, compare with a cosine. The cost of comparing n texts to m texts is `n + m` forward passes plus a matrix multiply - which is what makes clustering, dedup and search possible at all.

The pooling and normalisation rules are the same as `07_Feature_Extraction`: use the pooling the model was trained with (mean for MiniLM, CLS for bge, last token for Qwen3-Embedding), mask the padding, and L2-normalise so cosine similarity is a dot product.

Note what is *not* here: no prefix. These are symmetric comparisons, so both sides are encoded the same way. Retrieval prefixes (`"query: "`, bge's instruction) belong to the asymmetric case and would make the two sides of a symmetric pair inconsistent.

---

In [ ]:
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer


def pool(hidden, mask, how="mean"):
    "Collapse (batch, tokens, hidden) into (batch, hidden) using the model's own convention."
    if how == "cls":
        return hidden[:, 0]
    if how == "last":
        idx = mask.sum(dim=1) - 1
        return hidden[torch.arange(hidden.shape[0], device=hidden.device), idx]
    m = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)


def make_biencoder(model_id, how="mean", batch_size=64, max_length=128):
    "Return (embed_fn, model, tokenizer) for a bi-encoder and its pooling convention."
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=HF_CACHE)
    model = AutoModel.from_pretrained(model_id, dtype=dtype, cache_dir=HF_CACHE).to(device).eval()

    @torch.inference_mode()
    def embed(texts):
        vecs = []
        for i in range(0, len(texts), batch_size):
            enc = tok(texts[i:i + batch_size], padding=True, truncation=True,
                      max_length=max_length, return_tensors="pt").to(device)
            hidden = model(**enc).last_hidden_state
            v = pool(hidden, enc["attention_mask"], how).float()
            vecs.append(F.normalize(v, dim=-1).cpu())
        return torch.cat(vecs)

    return embed, model, tok


embed, model, tok = make_biencoder("sentence-transformers/all-MiniLM-L6-v2", how="mean")
vram("minilm loaded")

# The whole task, in three lines.
va, vb = embed(sent_a), embed(sent_b)
cos = (va * vb).sum(dim=1).tolist()
print(f"spearman {spearman(human, cos):.4f}   pearson {pearson(human, cos):.4f}")

# The pairs it gets most wrong, which is more informative than the correlation.
errors = sorted(zip(cos, human, sent_a, sent_b), key=lambda r: -(abs(r[0] - r[1])))
print("\nlargest disagreements with the human rating:")
for c, h, a, b in errors[:4]:
    print(f"  model {c:.2f} vs human {h:.2f}\n     {a}\n     {b}")

# The known blind spot, in one comparison.
neg = embed(["The deploy succeeded and all checks passed.",
             "The deploy did not succeed and the checks failed."])
print(f"\nnegated pair similarity: {(neg[0] @ neg[1]).item():.3f}  <- should be low; it is not")

## 9. Scores are not calibrated: look at the distribution first

The most common mistake with sentence similarity is treating the cosine as a probability. It is not, and the shape of the distribution shows why.

Below, the same model scores **matched pairs** (the real STS-B pairs) and **random pairs** (each sentence against an unrelated one). If cosine were a meaningful absolute scale, random pairs would sit near 0. They do not: most embedding spaces are anisotropic, so unrelated text lands at a substantial positive similarity, and the useful signal is the *separation* between the two distributions, not the value itself.

Consequences worth internalising:

- **The threshold must be learned, per model and per corpus.** Section 11 does it properly with a sweep.
- **A "0.7 similarity" claim in a bug report is meaningless** without the model name and the distribution.
- **Comparing two models by their raw scores is a category error.** Compare Spearman, or compare F1 after each has its own tuned threshold.

---

In [ ]:
import random

random.seed(0)
shuffled = sent_b[:]
random.shuffle(shuffled)
random_cos = (va * embed(shuffled)).sum(dim=1).tolist()

matched_mean = sum(cos) / len(cos)
random_mean = sum(random_cos) / len(random_cos)
print(f"matched pairs: mean cosine {matched_mean:.3f}, min {min(cos):.3f}, max {max(cos):.3f}")
print(f"random pairs : mean cosine {random_mean:.3f}, min {min(random_cos):.3f}, max {max(random_cos):.3f}")
print(f"\nseparation: {matched_mean - random_mean:.3f}  <- this is the signal, not the raw value")


def histogram(values, bins=20, lo=-0.2, hi=1.0):
    "Counts per bin, for the ECharts overlay below."
    edges = [lo + (hi - lo) * i / bins for i in range(bins + 1)]
    counts = [0] * bins
    for v in values:
        idx = min(int((v - lo) / (hi - lo) * bins), bins - 1)
        counts[max(idx, 0)] += 1
    return [f"{e:.2f}" for e in edges[:-1]], counts


labels_x, matched_counts = histogram(cos)
_, random_counts = histogram(random_cos)

from pyecharts import options as opts
from pyecharts.charts import Bar

bar = (
    Bar()
    .add_xaxis(labels_x)
    .add_yaxis("STS-B pairs", matched_counts)
    .add_yaxis("random pairs", random_counts)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="Cosine similarity distributions (all-MiniLM-L6-v2)",
            subtitle="random pairs do not sit at zero - this is why thresholds must be learned",
        ),
        xaxis_opts=opts.AxisOpts(name="cosine", axislabel_opts=opts.LabelOpts(rotate=45)),
        yaxis_opts=opts.AxisOpts(name="pairs"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
    )
)
bar.render_notebook()

## 10. The cross-encoder path

The same task with the two sentences in **one** sequence: `[CLS] sentence A [SEP] sentence B [SEP]`, full attention across both, and a single output logit as the score. Every token of A can attend to every token of B, which is information a bi-encoder structurally throws away.

It is reliably more accurate - and it produces no vectors, so nothing can be precomputed or indexed. Scoring one query against a million documents means a million forward passes.

The arithmetic below makes the trade-off concrete for a 10,000-document deduplication job: the bi-encoder encodes 10,000 texts and does a matrix multiply; the cross-encoder needs ~50 million forward passes. That is the difference between a coffee break and a week, and it is why the standard design is bi-encoder to shortlist, cross-encoder to decide.

---

In [ ]:
from transformers import AutoModelForSequenceClassification

del model, tok, embed
free_memory()
vram("before cross-encoder")

ce_id = "cross-encoder/stsb-roberta-base"
ce_tok = AutoTokenizer.from_pretrained(ce_id, cache_dir=HF_CACHE)
ce = AutoModelForSequenceClassification.from_pretrained(
    ce_id, dtype=dtype, cache_dir=HF_CACHE
).to(device).eval()
vram("cross-encoder loaded")
print("output logits:", ce.config.num_labels, "(one score, not a class distribution)")


@torch.inference_mode()
def cross_score(texts_a, texts_b, batch_size=32, max_length=128):
    "Score sentence pairs jointly. No vectors, nothing precomputable."
    scores = []
    for i in range(0, len(texts_a), batch_size):
        enc = ce_tok(texts_a[i:i + batch_size], texts_b[i:i + batch_size],
                     padding=True, truncation=True, max_length=max_length,
                     return_tensors="pt").to(device)
        logits = ce(**enc).logits.float().squeeze(-1)
        scores.extend(logits.tolist())
    return scores


print("\nwhat the model actually sees:")
print(ce_tok.decode(ce_tok(sent_a[0], sent_b[0])["input_ids"]))

t0 = time.perf_counter()
ce_scores = cross_score(sent_a, sent_b)
ce_seconds = time.perf_counter() - t0
print(f"\nspearman {spearman(human, ce_scores):.4f}   {ce_seconds:.1f}s for {len(sent_a)} pairs")

# The scaling wall, in numbers.
n = 10_000
print(f"\nall-pairs deduplication over {n:,} documents:")
print(f"  bi-encoder   : {n:,} encodings + one {n:,}x{n:,} matmul")
print(f"  cross-encoder: {n * (n - 1) // 2:,} forward passes "
      f"({(n * (n - 1) / 2) / (len(sent_a) / ce_seconds) / 3600:.1f} hours at this notebook's rate)")

del ce, ce_tok
free_memory()
vram("after cross-encoder")

## 11. Choosing a threshold, and using it: deduplication

Correlation is a research metric. What ships is a **decision**, and a decision needs a threshold.

The right way to pick one, done below:

1. Score labelled pairs with the model you will deploy.
2. Sweep every candidate cut and compute precision, recall and F1.
3. Choose from the **cost of each error type**, not from the F1 peak. In duplicate-merging, a false positive silently destroys data and a false negative merely leaves a duplicate; the correct threshold is far to the precision side of the F1 optimum.

The second half of the section applies the chosen threshold to actual deduplication over a small corpus, using the greedy first-wins algorithm that most production dedup uses - and it shows the practical detail that trips people up: at scale you do not compute all pairs, you generate candidates with an index or MinHash first, and reserve the scorer for those.

---

In [ ]:
import pandas as pd

embed, model, tok = make_biencoder("BAAI/bge-base-en-v1.5", how="cls")
vram("bge loaded")

bge_cos = (embed(sent_a) * embed(sent_b)).sum(dim=1).tolist()
sweep = threshold_sweep(bge_cos, is_duplicate)

best_f1 = max(sweep, key=lambda r: r["f1"])
# The threshold a data-destroying merge would actually justify: highest recall at >=99% precision.
safe = [r for r in sweep if r["precision"] >= 0.99]
safe_choice = max(safe, key=lambda r: r["recall"]) if safe else best_f1

print(f"spearman {spearman(human, bge_cos):.4f}")
print(f"best F1     : threshold {best_f1['threshold']:.3f}  P {best_f1['precision']:.3f}  "
      f"R {best_f1['recall']:.3f}  F1 {best_f1['f1']:.3f}")
print(f"P>=0.99 rule: threshold {safe_choice['threshold']:.3f}  P {safe_choice['precision']:.3f}  "
      f"R {safe_choice['recall']:.3f}  F1 {safe_choice['f1']:.3f}")
print("\nthe two differ by a lot - which is the point: F1 is not the objective, cost is")

# Deduplication with the chosen threshold, greedy first-wins.
CORPUS = [
    "How do I reset my password?",
    "I forgot my password, how can I reset it?",
    "What is the password reset process?",
    "How do I change my email address?",
    "The invoice for March has not arrived.",
    "I have not received the March invoice.",
    "Can I get a refund for my subscription?",
    "How do I cancel my subscription and get a refund?",
    "The dashboard is loading very slowly today.",
]
THRESHOLD = safe_choice["threshold"]

vecs = embed(CORPUS)
sims = vecs @ vecs.T
kept, duplicates = [], {}
for i in range(len(CORPUS)):
    match = next((j for j in kept if sims[i, j].item() >= THRESHOLD), None)
    if match is None:
        kept.append(i)
    else:
        duplicates.setdefault(match, []).append((i, sims[i, match].item()))

print(f"\ndeduplicating {len(CORPUS)} items at threshold {THRESHOLD:.3f}:")
for i in kept:
    print(f"  KEEP  {CORPUS[i]}")
    for j, s in duplicates.get(i, []):
        print(f"     dup ({s:.3f}) {CORPUS[j]}")
print(f"\n{len(CORPUS)} -> {len(kept)} items")

from pyecharts.charts import Line

line = (
    Line()
    .add_xaxis([f"{r['threshold']:.2f}" for r in sweep])
    .add_yaxis("precision", [round(r["precision"], 3) for r in sweep], is_smooth=True)
    .add_yaxis("recall", [round(r["recall"], 3) for r in sweep], is_smooth=True)
    .add_yaxis("F1", [round(r["f1"], 3) for r in sweep], is_smooth=True)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="Threshold sweep (bge-base-en-v1.5 on STS-B)",
                                  subtitle="pick the operating point from error cost, not from the F1 peak"),
        xaxis_opts=opts.AxisOpts(name="cosine threshold", axislabel_opts=opts.LabelOpts(rotate=45)),
        yaxis_opts=opts.AxisOpts(name="score", min_=0, max_=1),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
    )
)
line.render_notebook()

## 12. Head-to-head Benchmark

Four systems on the same 600 STS-B pairs: three bi-encoders and one cross-encoder, one model live at a time.

Read it as two separate rankings rather than one:

- **Among bi-encoders**, Spearman differences of a point or two on a saturated benchmark are noise; the columns that decide a deployment are pairs/second and vector size.
- **The cross-encoder is in a different regime.** It should top the correlation column, and its throughput number is per *pair*, not per text - so it cannot be compared with the bi-encoders' throughput at all. The all-pairs arithmetic in section 10 is the honest comparison.

---

In [ ]:
del embed, model, tok
free_memory()

BI_ENCODERS = [
    ("all-MiniLM-L6-v2", "sentence-transformers/all-MiniLM-L6-v2", "mean", 22),
    ("bge-base-en-v1.5", "BAAI/bge-base-en-v1.5", "cls", 109),
    ("qwen3-embedding-0.6b", "Qwen/Qwen3-Embedding-0.6B", "last", 596),
]

results = []
for name, model_id, how, params_m in BI_ENCODERS:
    embed_fn, mdl, tk = make_biencoder(model_id, how=how)
    t0 = time.perf_counter()
    scores = (embed_fn(sent_a) * embed_fn(sent_b)).sum(dim=1).tolist()
    elapsed = time.perf_counter() - t0
    sweep_i = threshold_sweep(scores, is_duplicate)
    results.append({
        "system": name, "type": "bi-encoder", "params_m": params_m,
        "dims": mdl.config.hidden_size,
        "spearman": round(spearman(human, scores), 4),
        "pearson": round(pearson(human, scores), 4),
        "best_f1": round(max(r["f1"] for r in sweep_i), 3),
        "pairs_per_sec": round(len(sent_a) / elapsed, 1),
    })
    print(results[-1])
    del embed_fn, mdl, tk    # free each model before loading the next so VRAM stays flat
    free_memory()

results.append({
    "system": "stsb-roberta-base", "type": "cross-encoder", "params_m": 125, "dims": 0,
    "spearman": round(spearman(human, ce_scores), 4),
    "pearson": round(pearson(human, ce_scores), 4),
    "best_f1": round(max(r["f1"] for r in threshold_sweep(ce_scores, is_duplicate)), 3),
    "pairs_per_sec": round(len(sent_a) / ce_seconds, 1),
})

vram("after benchmark")
df = pd.DataFrame(results).sort_values("spearman", ascending=False)
print("\nthroughput is per-pair for the cross-encoder and per-text for the bi-encoders")
df

In [ ]:
from pyecharts.charts import Bar

bar = (
    Bar()
    .add_xaxis([r["system"] for r in results])
    .add_yaxis("spearman x100", [round(r["spearman"] * 100, 1) for r in results])
    .add_yaxis("best F1 x100", [round(r["best_f1"] * 100, 1) for r in results])
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="STS-B test, 600 pairs",
            subtitle="RTX 3060 - the cross-encoder wins on quality and cannot be indexed",
        ),
        yaxis_opts=opts.AxisOpts(name="score", min_=0, max_=100),
        xaxis_opts=opts.AxisOpts(name="system", axislabel_opts=opts.LabelOpts(rotate=15)),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
    )
)
bar.render_notebook()

## 13. Interactive: compare your own sentences

Put your own sentences in and see the full similarity matrix, plus the pairs above your threshold. This is the cell people run on its own, so it opens with a `require(...)` guard naming what it needs from Setup rather than dying on a bare `NameError`.

The experiments that matter, because each is a documented failure mode:

- **Negation.** "The test passed" / "The test did not pass". Expect an uncomfortably high score.
- **Numbers and entities.** "Invoice 4471 is overdue" / "Invoice 8892 is overdue". Bi-encoders barely distinguish these, and for a ticketing system that is a serious bug.
- **High overlap, opposite meaning** - the PAWS pattern. "The cat chased the dog" / "The dog chased the cat".
- **Paraphrase with no shared words.** "It is raining heavily" / "There is a downpour outside". This is the case embeddings exist for, and where lexical matching fails.

If your task depends on the first three, add a cross-encoder rerank, or an NLI check (`04_Zero_Shot_Classification`), or a rule on the entities you care about.

---

In [ ]:
def require(*names):
    "Fail early and clearly if the notebook's setup / helper cells have not been run."
    missing = [n for n in names if n not in globals()]
    if missing:
        raise NameError(
            f"this demo needs {', '.join(missing)} from earlier in the notebook. "
            "Run the setup and helper cells first (Run > Run All Above Selected Cell)."
        )


require("device", "dtype", "HF_CACHE", "free_memory", "vram", "make_biencoder", "pool")

MY_SENTENCES = [
    "The test passed on the first run.",
    "The test did not pass on the first run.",
    "It is raining heavily.",
    "There is a downpour outside.",
    "The cat chased the dog.",
    "The dog chased the cat.",
    "Invoice 4471 is overdue.",
    "Invoice 8892 is overdue.",
]
MY_THRESHOLD = 0.75

# Re-runnable: this cell frees the model at the end, so guard the load or a second
# shift-enter raises NameError.
if "my_embed" not in globals():
    my_embed, my_model, my_tok = make_biencoder("BAAI/bge-base-en-v1.5", how="cls")
    vram("live model")

vecs = my_embed(MY_SENTENCES)
sims = vecs @ vecs.T

width = max(len(s) for s in MY_SENTENCES)
print(" " * (width + 2) + "".join(f"{i:6d}" for i in range(len(MY_SENTENCES))))
for i, s in enumerate(MY_SENTENCES):
    row = "".join(f"{sims[i, j].item():6.2f}" for j in range(len(MY_SENTENCES)))
    print(f"{s:<{width}}  {row}")

print(f"\npairs above {MY_THRESHOLD}:")
for i in range(len(MY_SENTENCES)):
    for j in range(i + 1, len(MY_SENTENCES)):
        if sims[i, j].item() >= MY_THRESHOLD:
            print(f"  {sims[i, j].item():.3f}  {MY_SENTENCES[i]!r} / {MY_SENTENCES[j]!r}")

del my_embed, my_model, my_tok
free_memory()
vram("final")

## 14. Going Further

- **Tune the threshold on your own labelled pairs, and re-tune it whenever the model changes.** A hundred hand-labelled pairs is enough, and it is the difference between a dedup system that works and one that silently merges unrelated records.
- **Two stages beat one.** Bi-encoder to shortlist (top 20-50), cross-encoder to decide. It gets you close to cross-encoder accuracy at bi-encoder cost, and it is the standard design - `11_Text_Ranking` builds it end to end.
- **Block before you score at scale.** MinHash/LSH for near-exact duplicates, or an ANN index for semantic candidates. All-pairs is quadratic and unnecessary.
- **Fine-tune with `MultipleNegativesRankingLoss`.** Give it (anchor, positive) pairs from your own data - duplicate tickets, linked issues, query/click logs - and let in-batch examples serve as negatives. A few thousand pairs typically beats any off-the-shelf model on your domain, and this is the single highest-value item in this list.
- **Add hard negatives.** Pairs that look similar and are not (same product, different version; same customer, different invoice) teach the distinctions that generic training data never covers.
- **Handle negation and numbers explicitly.** An NLI model for entailment/contradiction (`04_Zero_Shot_Classification`), or a rule that compares extracted entities and numbers (`01_Token_Classification`) before allowing a merge.
- **Cross-lingual matching needs a cross-lingual model.** `paraphrase-multilingual-*` and `multilingual-e5-*` place translations near each other; monolingual English models do not, and will silently rank a translation as unrelated.
- **Related notebooks.** `07_Feature_Extraction` (pooling, dimensions, storage arithmetic), `11_Text_Ranking` (asymmetric retrieval and the two-stage pipeline), `04_Zero_Shot_Classification` (NLI, which answers a different and often better question), `00_Text_Classification` (when the decision is a label, not a comparison).

---